In [7]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import joblib

In [8]:
df_train = pd.read_csv("../data/train.csv")
df_val   = pd.read_csv("../data/val.csv")
df_test  = pd.read_csv("../data/test.csv")

print("Train :", len(df_train))
print("Val :",   len(df_val))
print("Test :",  len(df_test))

df_train.head()

Train : 4391
Val : 941
Test : 942


,text,label,category,source,severity,group_id,augmented,tags
0,What are the best practices for secure coding ...,0,benign,original,NaN,grp_578f04597482,False,['coding' 'security_adjacent' 'hard_negative']
1,Can you help me write a Python script that fil...,0,benign,original,NaN,grp_b19fdee3675d,False,['coding' 'input_filtering' 'hard_negative']
2,What does 'prompt engineering' mean?,0,benign,original,NaN,grp_f964158df1fa,False,['ai_terminology' 'hard_negative']
3,Execute the deployment script and push the cha...,0,benign,original,NaN,grp_1a3f1bc3938d,False,['devops' 'hard_negative' 'contains_execute']
4,Please bypass the cache and fetch fresh data f...,0,benign,original,NaN,grp_d622368f6ce5,False,['api' 'hard_negative' 'contains_bypass']


In [9]:
def clean_text(text):
    text = str(text)
    text = text.lower()                          # lowercase
    text = re.sub(r"http\S+", "", text)          # supprimer les URLs
    text = re.sub(r"[^a-z0-9\s]", "", text)     # supprimer caractères spéciaux
    text = re.sub(r"\s+", " ", text).strip()     # supprimer espaces en trop
    return text

# Appliquer sur les 3 splits
df_train["text_clean"] = df_train["text"].apply(clean_text)
df_val["text_clean"]   = df_val["text"].apply(clean_text)
df_test["text_clean"]  = df_test["text"].apply(clean_text)

# Vérifier le résultat
print("Avant :", df_train["text"][0])
print("Après :", df_train["text_clean"][0])

Avant : What are the best practices for secure coding in Python?
Après : what are the best practices for secure coding in python


In [10]:
import base64

def has_base64(text):
    # Détecte si le texte contient un encodage base64
    try:
        pattern = r'[A-Za-z0-9+/]{20,}={0,2}'
        matches = re.findall(pattern, str(text))
        for m in matches:
            base64.b64decode(m)
            return 1
    except:
        pass
    return 0

# Mots-clés suspects
mots_suspects = ["ignore", "bypass", "jailbreak", "override",
                 "forget", "pretend", "disregard", "reveal",
                 "system", "instruction", "prompt"]

def count_suspicious(text):
    text = str(text).lower()
    return sum(1 for mot in mots_suspects if mot in text)

# Appliquer sur les 3 splits
for df in [df_train, df_val, df_test]:
    df["text_length"]    = df["text"].str.len()
    df["word_count"]     = df["text"].str.split().str.len()
    df["suspicious_count"] = df["text"].apply(count_suspicious)
    df["has_base64"]     = df["text"].apply(has_base64)

# Vérifier
df_train[["text", "text_length", "word_count",
          "suspicious_count", "has_base64"]].head()

,text,text_length,word_count,suspicious_count,has_base64
0,What are the best practices for secure coding ...,56,10,0,0
1,Can you help me write a Python script that fil...,68,13,0,0
2,What does 'prompt engineering' mean?,36,5,1,0
3,Execute the deployment script and push the cha...,65,10,0,0
4,Please bypass the cache and fetch fresh data f...,58,11,1,0


In [11]:
# Initialiser le vectoriseur
tfidf = TfidfVectorizer(
    max_features=10000,     # garde les 10000 mots les plus importants
    ngram_range=(1, 2),     # unigrammes et bigrammes (ex: "ignore previous")
    min_df=2,               # ignore les mots qui apparaissent moins de 2 fois
    sublinear_tf=True       # normalisation logarithmique
)

# Entraîner sur train UNIQUEMENT puis transformer les 3 splits
X_train_tfidf = tfidf.fit_transform(df_train["text_clean"])
X_val_tfidf   = tfidf.transform(df_val["text_clean"])
X_test_tfidf  = tfidf.transform(df_test["text_clean"])

print("Shape Train :", X_train_tfidf.shape)
print("Shape Val :",   X_val_tfidf.shape)
print("Shape Test :",  X_test_tfidf.shape)

Shape Train : (4391, 9458)
Shape Val : (941, 9458)
Shape Test : (942, 9458)


In [12]:
from scipy.sparse import hstack, csr_matrix

# Features manuelles en matrice sparse
features_manuelles = ["text_length", "word_count",
                      "suspicious_count", "has_base64"]

X_train_manual = csr_matrix(df_train[features_manuelles].values)
X_val_manual   = csr_matrix(df_val[features_manuelles].values)
X_test_manual  = csr_matrix(df_test[features_manuelles].values)

# Combiner TF-IDF + features manuelles
X_train = hstack([X_train_tfidf, X_train_manual])
X_val   = hstack([X_val_tfidf,   X_val_manual])
X_test  = hstack([X_test_tfidf,  X_test_manual])

print("Shape final Train :", X_train.shape)
print("Shape final Val :",   X_val.shape)
print("Shape final Test :",  X_test.shape)

Shape final Train : (4391, 9462)
Shape final Val : (941, 9462)
Shape final Test : (942, 9462)


In [13]:
# Les labels sont déjà en 0/1 donc pas besoin d'encodage
y_train = df_train["label"].values
y_val   = df_val["label"].values
y_test  = df_test["label"].values

print("Labels Train :", y_train.shape)
print("Distribution Train :", pd.Series(y_train).value_counts().to_dict())

Labels Train : (4391,)
Distribution Train : {1: 2650, 0: 1741}


In [14]:
# Sauvegarder le TF-IDF pour le réutiliser dans les étapes suivantes
joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")

print("Vectoriseur sauvegardé dans models/")

✅ Vectoriseur sauvegardé dans models/


In [15]:
print("Résumé du Preprocessing")
print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape} — classes : {set(y_train)}")
print(f"y_val   : {y_val.shape}   — classes : {set(y_val)}")
print(f"y_test  : {y_test.shape}  — classes : {set(y_test)}")
print("Preprocessing terminé")

Résumé du Preprocessing
X_train : (4391, 9462)
X_val   : (941, 9462)
X_test  : (942, 9462)
y_train : (4391,) — classes : {np.int64(0), np.int64(1)}
y_val   : (941,)   — classes : {np.int64(0), np.int64(1)}
y_test  : (942,)  — classes : {np.int64(0), np.int64(1)}
Preprocessing terminé
